# Climate Map Generator
**Output:** Generates 8 interactive HTML maps in the `../maps/` folder.
**Color Coding:** 0 (Red/Risk) to 11 (Green/Safe)

In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import os

# 1. SETUP OUTPUT FOLDER
output_dir = "../maps"
os.makedirs(output_dir, exist_ok=True)
pio.renderers.default = None

# 2. LOAD DATA
df = pd.read_csv('../data/FullJoin3_with_climate_ratings-proofed.csv')

# 3. CLEANING & FORMATTING
df['LocationCoordY_fixed'] = df[['LocationCoordX', 'LocationCoordY']].min(axis=1) # Longitude
df['LocationCoordX_fixed'] = df[['LocationCoordX', 'LocationCoordY']].max(axis=1) # Latitude

exclude_locs = ['Nursery', 'Nitobe Memorial Garden', 'Nitobe']
df_clean = df[~df['LocationName'].isin(exclude_locs)].copy()
# Filter outlier coordinates
df_clean = df_clean[df_clean['LocationCoordY_fixed'] < -123.23].copy() 

# Apply Jitter
jitter_strength = 0.00006 
np.random.seed(42) 
df_clean['LocationCoordX_Jittered'] = df_clean['LocationCoordX_fixed'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(df_clean))
df_clean['LocationCoordY_Jittered'] = df_clean['LocationCoordY_fixed'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(df_clean))

# Renaming & Melting
rename_dict = {
    'climate_rating_current_sheet': 'Current',
    'climate_rating_emissions_limited_2050': '2050',
    'climate_rating_business_as_usual_2090': '2090',
    'climate_rating_bau_plus_1degree_2090': '2090_plus_1_degree',
    'GenusSpecies': 'Taxon'
}
df_ready = df_clean.rename(columns=rename_dict)

lifeform_mapping = {
    'Shrub': 'Woody', 'Tree': 'Woody', 'Shrub or Tree': 'Woody',
    'Climber_Liana_Vine': 'Woody', 'Herbaceous Perennial': 'Perennial',
    'Bulb, Corm, or Tuber': 'Perennial', 'Annual': 'Short-lived',
    'Biennial': 'Short-lived', 'Habit unknown': 'Unknown'
}
df_ready['LifeForm'] = df_ready['LifeForm'].map(lifeform_mapping).fillna('Unknown')

long_format_df = pd.melt(
    df_ready,
    id_vars=['ItemAccNoFull', 'LocationCoordX_Jittered', 'LocationCoordY_Jittered', 'Taxon', 'LifeForm'],
    value_vars=['Current', '2050', '2090', '2090_plus_1_degree'],
    var_name='Era',
    value_name='ClimateRating'
).dropna(subset=['ClimateRating'])

plot_df = long_format_df[long_format_df['LifeForm'].isin(['Woody', 'Perennial'])].copy()

# --- CRITICAL FIX: GEOMETRIC CENTERING ---
# We calculate the center based on the Min/Max of the ENTIRE garden.
# This ensures the "Long" Asian Garden is included, even if fewer plants are there.
global_lat = (df_clean['LocationCoordX_Jittered'].max() + df_clean['LocationCoordX_Jittered'].min()) / 2
global_lon = (df_clean['LocationCoordY_Jittered'].max() + df_clean['LocationCoordY_Jittered'].min()) / 2

# 4. GENERATE MAPS LOOP
era_labels = {'Current': 'Current Conditions', '2050': '2050 (Emissions Limited)', '2090': '2090 (BAU)', '2090_plus_1_degree': '2090 (+1C)'}

print("Generating 8 Map Files (Height=1000px, Geometric Center)...")

for lifeform in ['Woody', 'Perennial']:
    for era in ['Current', '2050', '2090', '2090_plus_1_degree']:
        subset = plot_df[(plot_df['LifeForm'] == lifeform) & (plot_df['Era'] == era)]
        
        fig = go.Figure()
        
        fig.add_trace(go.Scattermapbox(
            lat=subset['LocationCoordX_Jittered'],
            lon=subset['LocationCoordY_Jittered'],
            mode='markers',
            marker=dict(
                size=9,             # Big dots
                color=subset['ClimateRating'],
                colorscale='RdYlGn',
                cmin=0, cmax=11,
                opacity=0.9,
                showscale=True,
                colorbar=dict(title="Suitability")
            ),
            text=subset['Taxon'],
            hoverinfo='text'
        ))
            
        fig.update_layout(
            mapbox=dict(
                style="carto-positron", 
                zoom=15.8,
                # Use the GLOBAL center so the map doesn't jump
                center=dict(lat=global_lat, lon=global_lon)
            ),
            title_text=f"{lifeform} - {era_labels[era]}",
            height=1000, # <-- Taller map box
            margin={"r":0,"t":40,"l":0,"b":0}
        )
        
        filepath = os.path.join(output_dir, f"{lifeform}_{era}.html")
        fig.write_html(filepath, include_plotlyjs='cdn')
        print(f"Created: {filepath}")

Generating 8 Map Files (Height=1000px, Geometric Center)...
Created: ../maps/Woody_Current.html
Created: ../maps/Woody_2050.html
Created: ../maps/Woody_2090.html
Created: ../maps/Woody_2090_plus_1_degree.html
Created: ../maps/Perennial_Current.html
Created: ../maps/Perennial_2050.html
Created: ../maps/Perennial_2090.html
Created: ../maps/Perennial_2090_plus_1_degree.html


/var/folders/jk/2h_df7j95595lz7z2w2kt4kr0000gn/T/ipykernel_49003/2865230530.py:75: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

